# AI Agents - UC2 DQN
## Lab: Neural Network Fundamentals

Before training a DQN agent (see [rl101-naive-dqn.ipynb](./rl101-naive-dqn.ipynb)), it helps to understand the **neural network** that sits at its core — the `QNetwork`. This notebook builds that understanding from the ground up, with nothing but `numpy` and basic math:

1. **Modeling a single neuron** — input data, weights, bias.
2. **Activation functions** — why non-linearity matters.
3. **Why one neuron isn't enough** — the XOR problem.
4. **Training a small network** — forward pass, loss, and backpropagation, all derived and coded by hand.
5. **Experiments** — change the learning rate, network size, and activation function, and watch how training behaves differently.
6. **Bridging to PyTorch** — the same network, trained with `autograd`, to connect this to the `QNetwork` used in the DQN notebooks.

## Setup

In [ ]:
import os

# Must be set before numpy/torch load to avoid duplicate OpenMP runtime crash on macOS
# (conda's numpy/llvm-openmp and PyTorch's bundled libiomp5 both try to init OpenMP).
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

## 1. Modeling a single neuron

A single artificial neuron takes some **input data** `x` (a vector), multiplies it by a learned **weight vector** `w`, adds a learned **bias** `b`, and passes the result through an **activation function** `f`:

$$z = w \cdot x + b \qquad y = f(z)$$

- `x`: the input features (e.g. for CartPole this would be `[cart_pos, cart_vel, pole_angle, pole_ang_vel]`).
- `w`: one learnable weight per input feature — how much each input contributes.
- `b`: a learnable bias — shifts the weighted sum, letting the neuron activate even when all inputs are zero.
- `f`: the activation function (see section 2).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

inputs = ["x1", "x2", "x3"]
weights = ["w1", "w2", "w3"]
input_y = [2.5, 1.5, 0.5]

for x_label, w_label, y in zip(inputs, weights, input_y):
    ax.add_patch(patches.Circle((0.5, y), 0.25, facecolor="#a6cee3", edgecolor="black", zorder=3))
    ax.text(0.5, y, x_label, ha="center", va="center", fontsize=11, zorder=4)
    ax.annotate("", xy=(2.2, 1.5), xytext=(0.78, y), arrowprops=dict(arrowstyle="->", color="gray", lw=1.5))
    mid_x, mid_y = (0.78 + 2.2) / 2, (y + 1.5) / 2
    ax.text(mid_x, mid_y + 0.12, w_label, color="black", fontsize=10, ha="center")

# Bias is modeled as a constant "1" input with its own learned weight b
ax.add_patch(patches.Circle((0.5, -0.6), 0.25, facecolor="#fdbf6f", edgecolor="black", zorder=3))
ax.text(0.5, -0.6, "1", ha="center", va="center", fontsize=11, zorder=4)
ax.annotate("", xy=(2.2, 1.5), xytext=(0.78, -0.6), arrowprops=dict(arrowstyle="->", color="gray", lw=1.5))
ax.text(1.45, -0.3, "b", fontsize=10, ha="center")

# Weighted-sum node
ax.add_patch(patches.Circle((2.5, 1.5), 0.35, facecolor="#b2df8a", edgecolor="black", zorder=3))
ax.text(2.5, 1.5, r"$\Sigma$", ha="center", va="center", fontsize=14, zorder=4)

ax.annotate("", xy=(4.0, 1.5), xytext=(2.85, 1.5), arrowprops=dict(arrowstyle="->", color="black", lw=2))
ax.text(3.4, 1.8, "z = w\u00b7x + b", fontsize=9, ha="center")

# Activation box
ax.add_patch(patches.FancyBboxPatch((3.9, 1.2), 0.9, 0.6, boxstyle="round,pad=0.05",
                                     facecolor="#fb9a99", edgecolor="black", zorder=3))
ax.text(4.35, 1.5, "f(z)", ha="center", va="center", fontsize=12, zorder=4)

ax.annotate("", xy=(6.0, 1.5), xytext=(4.8, 1.5), arrowprops=dict(arrowstyle="->", color="black", lw=2))
ax.text(6.2, 1.5, "y", fontsize=13, va="center")

ax.set_xlim(-0.2, 6.6)
ax.set_ylim(-1.2, 3.2)
ax.set_aspect("equal")
ax.axis("off")
ax.set_title("A single neuron: weighted sum + bias, then an activation function")
plt.tight_layout()
plt.show()

In [ ]:
def neuron(x: np.ndarray, w: np.ndarray, b: float, activation) -> float:
    """A single neuron: z = w . x + b, y = activation(z)."""
    z = np.dot(w, x) + b
    return activation(z)


def identity(z):
    return z


# Quick example: 3 inputs, hand-picked weights and bias
x_example = np.array([1.0, 2.0, -1.0])
w_example = np.array([0.5, -0.3, 0.8])
b_example = 0.1

z_example = np.dot(w_example, x_example) + b_example
y_example = neuron(x_example, w_example, b_example, identity)
print(f"x = {x_example}")
print(f"w = {w_example}")
print(f"b = {b_example}")
print(f"z = w\u00b7x + b = {z_example:.3f}")
print(f"y = f(z) = {y_example:.3f}  (using the identity activation, f(z) = z)")

## 2. Activation functions

Without an activation function (or with a linear one, like `identity` above), stacking neurons would still only ever produce a linear function of the input — no matter how many layers you add. **Non-linear activation functions** are what let networks approximate complex, non-linear relationships.

- **Step**: the original 1950s perceptron activation — fires `1` or `0`. Its derivative is zero almost everywhere, so it cannot be used with gradient-based training (included here only for historical context).
- **Sigmoid**: squashes any real number into `(0, 1)` — useful for outputting probabilities. Saturates (flattens) for large `|z|`, which can slow down learning.
- **Tanh**: like sigmoid but squashes into `(-1, 1)` and is zero-centered, which often helps optimization.
- **ReLU**: `max(0, z)` — the most common choice in modern deep networks (including the `QNetwork` in the DQN notebooks). Cheap to compute and does not saturate for `z > 0`.

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def sigmoid_deriv(z):
    s = sigmoid(z)
    return s * (1 - s)


def tanh(z):
    return np.tanh(z)


def tanh_deriv(z):
    return 1 - np.tanh(z) ** 2


def relu(z):
    return np.maximum(0, z)


def relu_deriv(z):
    return (z > 0).astype(float)


def step(z):
    return (z >= 0).astype(float)


# Hidden-layer activations usable in the network we build later (output layer always uses sigmoid)
ACTIVATIONS = {
    "sigmoid": (sigmoid, sigmoid_deriv),
    "tanh": (tanh, tanh_deriv),
    "relu": (relu, relu_deriv),
}

z = np.linspace(-6, 6, 400)
funcs = [("Step", step, None), ("Sigmoid", sigmoid, sigmoid_deriv),
         ("Tanh", tanh, tanh_deriv), ("ReLU", relu, relu_deriv)]

fig, axes = plt.subplots(2, 4, figsize=(14, 6), sharex=True)
for col, (name, f, df) in enumerate(funcs):
    axes[0, col].plot(z, f(z), color="steelblue")
    axes[0, col].set_title(name)
    axes[0, col].axhline(0, color="gray", lw=0.5)
    axes[0, col].axvline(0, color="gray", lw=0.5)
    if df is not None:
        axes[1, col].plot(z, df(z), color="darkorange")
    else:
        axes[1, col].text(0.5, 0.5, "derivative is 0\n(can't train with gradients)",
                           ha="center", va="center", transform=axes[1, col].transAxes, fontsize=8)
    axes[1, col].axhline(0, color="gray", lw=0.5)
    axes[1, col].axvline(0, color="gray", lw=0.5)
    axes[1, col].set_xlabel("z")

axes[0, 0].set_ylabel("f(z)")
axes[1, 0].set_ylabel("f'(z)")
plt.tight_layout()
plt.show()

## 3. Why one neuron isn't enough — the XOR problem

A single neuron (with any activation above) can only separate its inputs with a **straight line** (or hyperplane, in higher dimensions) — it is a *linear* classifier. The **XOR** function is the classic example of a simple pattern that is **not linearly separable**:

| x1 | x2 | XOR(x1, x2) |
|----|----|----|
| 0  | 0  | 0  |
| 0  | 1  | 1  |
| 1  | 0  | 1  |
| 1  | 1  | 0  |

No single straight line can separate the `0`s from the `1`s on this truth table. We will use XOR as our toy training problem for the rest of the notebook, because it is small (4 examples), trains in a fraction of a second, and clearly demonstrates *why* we need a hidden layer.

In [ ]:
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y = np.array([[0], [1], [1], [0]], dtype=float)

print("XOR truth table:")
for xi, yi in zip(X, y):
    print(f"  x={xi} -> y={yi[0]:.0f}")

In [ ]:
def train_single_neuron(X, y, lr=0.5, epochs=5000, seed=0):
    """Logistic regression: one neuron, no hidden layer, sigmoid activation, MSE loss."""
    rng = np.random.default_rng(seed)
    W = rng.normal(0, 1, size=(X.shape[1], 1))
    b = np.zeros(1)
    losses = []
    for _ in range(epochs):
        Z = X @ W + b
        A = sigmoid(Z)
        losses.append(np.mean((A - y) ** 2))

        dA = 2 * (A - y) / X.shape[0]
        dZ = dA * sigmoid_deriv(Z)
        dW = X.T @ dZ
        db = dZ.sum(axis=0)
        W -= lr * dW
        b -= lr * db
    return W, b, losses


W_single, b_single, single_losses = train_single_neuron(X, y)
preds_single = sigmoid(X @ W_single + b_single)

plt.figure(figsize=(5, 3))
plt.plot(single_losses)
plt.title("Single neuron (no hidden layer) trying to learn XOR")
plt.xlabel("epoch")
plt.ylabel("MSE loss")
plt.show()

print("Predictions from a single neuron (no hidden layer) -- notice it can't tell all 4 cases apart:")
for xi, yi, pi in zip(X, y, preds_single):
    print(f"  x={xi} -> target={yi[0]:.0f}, predicted={pi[0]:.3f}")

As expected, the loss plateaus and the predictions stay close to `0.5` for at least some inputs — a single neuron cannot represent a decision boundary that solves XOR, no matter how long you train it or what learning rate you pick.

## 4. Building a small network (2 → H → 1)

Stacking a **hidden layer** of `H` neurons between the input and the output lets the network combine several linear decision boundaries (bent by the activation function) into a non-linear one that *can* solve XOR.

$$Z_1 = X W_1 + b_1 \qquad A_1 = f_{\text{hidden}}(Z_1)$$
$$Z_2 = A_1 W_2 + b_2 \qquad A_2 = \text{sigmoid}(Z_2)$$

- `X`: input batch, shape `(N, 2)` for `N` examples.
- `W1`, `b1`: hidden layer's weights `(2, H)` and biases `(H,)`.
- `W2`, `b2`: output layer's weights `(H, 1)` and bias `(1,)`.
- `A2`: the network's prediction, squashed to `(0, 1)` by a sigmoid output (since our XOR targets are `0` or `1`).

In [ ]:
def init_params(input_size, hidden_size, output_size, seed=0):
    rng = np.random.default_rng(seed)
    W1 = rng.normal(0, 1, size=(input_size, hidden_size))
    b1 = np.zeros(hidden_size)
    W2 = rng.normal(0, 1, size=(hidden_size, output_size))
    b2 = np.zeros(output_size)
    return W1, b1, W2, b2


def forward(X, W1, b1, W2, b2, hidden_activation="sigmoid"):
    act, _ = ACTIVATIONS[hidden_activation]
    Z1 = X @ W1 + b1
    A1 = act(Z1)
    Z2 = A1 @ W2 + b2
    A2 = sigmoid(Z2)  # output layer always uses sigmoid: squashes to [0, 1] for our binary targets
    cache = (Z1, A1, Z2, A2)
    return A2, cache

## 5. Loss function

To train the network we need a single number that measures "how wrong" its predictions are, so that gradient descent has something to minimize. We use the **mean squared error (MSE)** between predictions `A2` and targets `y`:

$$L = \frac{1}{N} \sum_i (A_2^{(i)} - y^{(i)})^2$$

(Binary cross-entropy is the more common choice for classification in practice, but MSE keeps the derivative — and the backprop code below — simple and easy to follow by hand.)

## 6. Backpropagation — deriving and implementing the gradients

Training means repeatedly nudging every weight and bias in the direction that reduces the loss the most: **gradient descent**. The gradients are computed with the **chain rule**, propagating the error backwards from the output to the input — hence "backpropagation".

For our 2-layer network with MSE loss and a sigmoid output:

$$\frac{\partial L}{\partial A_2} = \frac{2}{N}(A_2 - y)$$
$$\frac{\partial L}{\partial Z_2} = \frac{\partial L}{\partial A_2} \odot \text{sigmoid}'(Z_2)$$
$$\frac{\partial L}{\partial W_2} = A_1^\top \frac{\partial L}{\partial Z_2} \qquad \frac{\partial L}{\partial b_2} = \sum \frac{\partial L}{\partial Z_2}$$
$$\frac{\partial L}{\partial A_1} = \frac{\partial L}{\partial Z_2} W_2^\top$$
$$\frac{\partial L}{\partial Z_1} = \frac{\partial L}{\partial A_1} \odot f_{\text{hidden}}'(Z_1)$$
$$\frac{\partial L}{\partial W_1} = X^\top \frac{\partial L}{\partial Z_1} \qquad \frac{\partial L}{\partial b_1} = \sum \frac{\partial L}{\partial Z_1}$$

Then every parameter is updated by taking a small step (the **learning rate** `lr`) opposite to its gradient: `W -= lr * dW`.

In [ ]:
def backward(X, y, W2, cache, hidden_activation="sigmoid"):
    _, dact = ACTIVATIONS[hidden_activation]
    Z1, A1, Z2, A2 = cache
    n = X.shape[0]

    dA2 = 2 * (A2 - y) / n          # dL/dA2 -- derivative of the MSE loss
    dZ2 = dA2 * sigmoid_deriv(Z2)   # dL/dZ2 -- chain rule through the output sigmoid
    dW2 = A1.T @ dZ2                # dL/dW2
    db2 = dZ2.sum(axis=0)           # dL/db2

    dA1 = dZ2 @ W2.T                # back-propagate the error through W2
    dZ1 = dA1 * dact(Z1)            # dL/dZ1 -- chain rule through the hidden activation
    dW1 = X.T @ dZ1                 # dL/dW1
    db1 = dZ1.sum(axis=0)           # dL/db1

    return dW1, db1, dW2, db2


def train(X, y, hidden_size=4, lr=1.0, epochs=10000, hidden_activation="sigmoid", seed=0):
    W1, b1, W2, b2 = init_params(X.shape[1], hidden_size, y.shape[1], seed=seed)
    losses = []
    for _ in range(epochs):
        A2, cache = forward(X, W1, b1, W2, b2, hidden_activation)
        losses.append(np.mean((A2 - y) ** 2))

        dW1, db1, dW2, db2 = backward(X, y, W2, cache, hidden_activation)

        # Gradient descent update: take a small step opposite the gradient
        W1 -= lr * dW1
        b1 -= lr * db1
        W2 -= lr * dW2
        b2 -= lr * db2

    return W1, b1, W2, b2, losses

In [ ]:
W1, b1, W2, b2, losses = train(X, y, hidden_size=4, lr=1.0, epochs=10000, hidden_activation="sigmoid", seed=0)

plt.figure(figsize=(5, 3))
plt.plot(losses)
plt.title("2 \u2192 4 \u2192 1 network learning XOR")
plt.xlabel("epoch")
plt.ylabel("MSE loss")
plt.show()

preds, _ = forward(X, W1, b1, W2, b2, "sigmoid")
print(f"Final loss: {losses[-1]:.5f}")
print("Predictions after training:")
for xi, yi, pi in zip(X, y, preds):
    print(f"  x={xi} -> target={yi[0]:.0f}, predicted={pi[0]:.3f}")

With a hidden layer, the network drives the loss close to zero and separates all four XOR cases correctly -- something the single neuron in section 3 could never do.

## 7. Experiments — change the variables, watch training change

Now that we have a working `train()` function, we can run small experiments to build intuition for how each hyperparameter affects learning. Each one re-trains the network from scratch with everything else held fixed.

### Experiment 1 — learning rate

Too small and training crawls; a good value converges quickly and smoothly; too large and the neurons can saturate and get stuck (loss flatlines at `0.5`, the worst possible score for this task).

In [ ]:
learning_rates = [0.05, 0.5, 2.0, 100.0]

plt.figure(figsize=(7, 4))
for lr in learning_rates:
    _, _, _, _, lr_losses = train(X, y, hidden_size=4, lr=lr, epochs=2000, hidden_activation="sigmoid", seed=0)
    plt.plot(lr_losses, label=f"lr={lr}")
    print(f"lr={lr:6} -> final loss after 2000 epochs: {lr_losses[-1]:.5f}")

plt.yscale("log")
plt.xlabel("epoch")
plt.ylabel("MSE loss (log scale)")
plt.title("Effect of learning rate")
plt.legend()
plt.show()

**What to notice:** `lr=0.05` barely moves in 2000 epochs (too slow). `lr=0.5` and `lr=2.0` converge nicely (good range). `lr=100.0` jumps so far on the very first update that the sigmoids saturate — their gradients vanish, and the network gets permanently stuck at loss `0.5`, unable to learn at all. This is the classic "learning rate too high" failure mode.

### Experiment 2 — hidden layer size

How many hidden neurons does XOR actually need?

In [ ]:
hidden_sizes = [1, 2, 4, 8]

plt.figure(figsize=(7, 4))
for hs in hidden_sizes:
    _, _, _, _, hs_losses = train(X, y, hidden_size=hs, lr=1.0, epochs=10000, hidden_activation="sigmoid", seed=0)
    plt.plot(hs_losses, label=f"hidden={hs}")
    print(f"hidden_size={hs} -> final loss after 10000 epochs: {hs_losses[-1]:.5f}")

plt.yscale("log")
plt.xlabel("epoch")
plt.ylabel("MSE loss (log scale)")
plt.title("Effect of hidden layer size")
plt.legend()
plt.show()

**What to notice:** with only **1** hidden neuron the network still cannot solve XOR (not enough capacity to combine two bent decision boundaries) and the loss plateaus, similar to the single-neuron case in section 3. With **2 or more** hidden neurons, the network easily reaches near-zero loss -- 2 is the minimum needed to represent XOR exactly.

### Experiment 3 — activation function

Same architecture, same learning rate, same seed -- only the hidden layer's activation function changes.

In [ ]:
plt.figure(figsize=(7, 4))
for act_name in ["sigmoid", "tanh", "relu"]:
    _, _, _, _, act_losses = train(X, y, hidden_size=4, lr=1.0, epochs=10000, hidden_activation=act_name, seed=0)
    plt.plot(act_losses, label=act_name)
    print(f"activation={act_name:8} -> final loss after 10000 epochs: {act_losses[-1]:.5f}")

plt.yscale("log")
plt.xlabel("epoch")
plt.ylabel("MSE loss (log scale)")
plt.title("Effect of hidden activation function")
plt.legend()
plt.show()

**What to notice:** `sigmoid` and `relu` both solve XOR well with this seed. `tanh` can get stuck at a noticeably higher loss -- it has landed in a **local minimum** (a symmetric solution where two of the four predictions sit at `0.5`). This is a real, well-known phenomenon in small networks: the random initial weights (controlled by `seed`) determine which solution gradient descent finds. Try changing `seed=0` to a different number in the `tanh` run above and re-plotting -- you'll often see it converge perfectly instead.

### Ideas to try yourself

- **Random seed**: re-run section 6's training with `seed=1, 2, 3, ...` -- does it always converge equally fast?
- **Weight initialization scale**: in `init_params`, change `rng.normal(0, 1, ...)` to `rng.normal(0, 0.01, ...)` or `rng.normal(0, 10, ...)` -- very small or very large initial weights can slow down or destabilize training.
- **Number of epochs**: train for only `200` epochs instead of `10000` -- how far from converged is the network?
- **Loss function**: swap the MSE loss for binary cross-entropy (`-mean(y*log(A2) + (1-y)*log(1-A2))`) and re-derive `dA2` -- does it train faster?
- **A second hidden layer**: extend `forward`/`backward`/`init_params` to a `2 \u2192 H1 \u2192 H2 \u2192 1` network.

## 8. Bridging to PyTorch

Everything above -- the forward pass, the hand-derived gradients, the parameter updates -- is exactly what a deep learning framework like **PyTorch** automates:

- `nn.Linear` replaces our manual `X @ W + b`.
- `loss.backward()` (**autograd**) replaces our manual `backward()` function -- it computes every gradient automatically via the same chain rule.
- `optimizer.step()` replaces our manual `W -= lr * dW` update.

This is precisely the pattern used by the `QNetwork` and training loop in [rl101-naive-dqn.ipynb](./rl101-naive-dqn.ipynb). Let's rebuild the same XOR network in PyTorch and confirm it learns the same thing.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(0)


class TinyXORNet(nn.Module):
    def __init__(self, hidden_size: int = 4):
        super().__init__()
        self.hidden = nn.Linear(2, hidden_size)
        self.output = nn.Linear(hidden_size, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = torch.sigmoid(self.hidden(x))
        return torch.sigmoid(self.output(h))


X_t = torch.tensor(X, dtype=torch.float32)
y_t = torch.tensor(y, dtype=torch.float32)

net = TinyXORNet(hidden_size=4)
optimizer = optim.SGD(net.parameters(), lr=1.0)
loss_fn = nn.MSELoss()

torch_losses = []
for epoch in range(10000):
    pred = net(X_t)
    loss = loss_fn(pred, y_t)

    optimizer.zero_grad()
    loss.backward()   # autograd computes every gradient we derived by hand in section 6
    optimizer.step()  # applies the same W -= lr * dW update rule

    torch_losses.append(loss.item())

plt.figure(figsize=(5, 3))
plt.plot(torch_losses)
plt.title("Same network, trained with PyTorch autograd")
plt.xlabel("epoch")
plt.ylabel("MSE loss")
plt.show()

print(f"Final loss: {torch_losses[-1]:.5f}")
print("PyTorch predictions after training:")
with torch.no_grad():
    preds_t = net(X_t)
for xi, yi, pi in zip(X, y, preds_t):
    print(f"  x={xi} -> target={yi[0]:.0f}, predicted={pi.item():.3f}")

## Wrap-up

The hand-written `train()` function and the PyTorch version converge to essentially the same solution -- because they are doing the same math. The only difference is who computes the gradients: you, by hand (section 6), or `autograd`, automatically.

In [rl101-naive-dqn.ipynb](./rl101-naive-dqn.ipynb), `QNetwork` is a bigger version of `TinyXORNet` above (4 inputs instead of 2, 2 hidden layers of 128 neurons instead of 1 of 4, ReLU instead of sigmoid, and 2 outputs -- one Q-value per action, instead of 1). The training loop there follows the exact same recipe you just implemented by hand: forward pass, compute a loss, `loss.backward()`, `optimizer.step()`.